In [1]:
##rate coding vectorized and optimized with original parameters

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display
from itertools import count
import wandb
import os

# Initialize WandB
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="count-rate_c4")

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Connect Four Environment Class (Unchanged)
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self):
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id): return True
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id): return True
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]): return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]): return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            print('Move is invalid')
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

# Replay Memory Class (Unchanged)
class ReplayMemory:
    def __init__(self, capacity=100000):
        self.memory = []
        self.capacity = capacity
        self.position = 0

    def dump(self, transition):
        if len(self.memory) < self.capacity: self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

# Surrogate Gradient Spike Function (Unchanged)
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN) Class (Unchanged)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size,
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        torch.manual_seed(seed)
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        mem_rec = []
        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        q_values = mem_rec[-1]
        return q_values, mem_rec, spk

# DSQN Agent Class (Updated)
class DSQN:
    def __init__(self, discount_factor, dsnn_config):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        self.board_size = 6 * 7

    def rate_encode_single(self, state):
        """Original iterative encoding for a single state (used in select_action)."""
        state_tensor = torch.tensor(state, dtype=torch.float, device=device)
        encoded = torch.zeros(self.simulation_time, self.board_size, device=device)
        state_flat = state_tensor.flatten()
        for pos, val in enumerate(state_flat):
            if val == 0: spike_prob = 0.0
            elif val == 1: spike_prob = 3.0 / self.simulation_time
            elif val == 2: spike_prob = 1.0
            encoded[:, pos] = torch.bernoulli(torch.full((self.simulation_time,), spike_prob, device=device))
        return encoded.unsqueeze(0)

    def rate_encode_batch(self, states_batch):
        """Vectorized rate encoding for a batch of states."""
        batch_size = states_batch.size(0)
        states_flat = states_batch.view(batch_size, -1)
        spike_probs = torch.zeros_like(states_flat, dtype=torch.float, device=device)
        spike_probs[states_flat == 1] = 3.0 / self.simulation_time
        spike_probs[states_flat == 2] = 1.0
        spike_probs_over_time = spike_probs.unsqueeze(2).expand(-1, -1, self.simulation_time)
        encoded = torch.bernoulli(spike_probs_over_time)
        return encoded.permute(0, 2, 1)

    def select_action(self, state, available_actions, training=True, steps_done=None):
        encoded_state = self.rate_encode_single(state)
        if training:
            if steps_done is None: raise ValueError("steps_done is required")
            eps_threshold = EPS_END + (EPS_START - EPS_END) * np.exp(-steps_done / EPS_DECAY)
        else:
            eps_threshold = 0
        if random.random() > eps_threshold:
            with torch.no_grad():
                q_values, _, spk = self.training_net(encoded_state)
                # Sum spikes across hidden layers for the last timestep, as in Tic-Tac-Toe
                step_spike_count = sum(layer_spikes[-1].sum().item() for layer_spikes in spk[:-1] if layer_spikes)
                total_spike_count[0] += step_spike_count
                episode_count_for_avg_spikes[0] += 1
                q_values = q_values.flatten()
                valid_q = q_values[available_actions]
                return available_actions[torch.argmax(valid_q).item()]
        return random.choice(available_actions)

    def optimize_model(self, memory, episode):
        if len(memory) < self.batch_size:
            return 0.0, 0
        
        transitions = memory.sample(self.batch_size)
        state_batch, action_batch, reward_batch, next_state_batch = zip(*transitions)
        
        state_batch = torch.tensor(np.array(state_batch), dtype=torch.float, device=device)
        action_batch = torch.tensor(action_batch, dtype=torch.long, device=device)
        reward_batch = torch.tensor(reward_batch, dtype=torch.float, device=device)
        non_final_mask = torch.tensor([s is not None for s in next_state_batch], device=device)
        non_final_next_states = [s for s in next_state_batch if s is not None]
        non_final_next_states = torch.tensor(np.array(non_final_next_states), dtype=torch.float, device=device) if non_final_next_states else None
        
        state_batch_encoded = self.rate_encode_batch(state_batch)
        current_q, _, _ = self.training_net(state_batch_encoded)
        
        valid_action_mask = action_batch != -1
        current_q_selected = torch.zeros(self.batch_size, device=device)
        if valid_action_mask.any():
            valid_indices = valid_action_mask.nonzero(as_tuple=True)[0]
            valid_actions = action_batch[valid_action_mask].unsqueeze(1)
            current_q_selected[valid_indices] = current_q[valid_indices].gather(1, valid_actions).squeeze(1)
        
        next_state_values = torch.zeros(self.batch_size, device=device)
        if non_final_next_states is not None:
            next_state_batch_encoded = self.rate_encode_batch(non_final_next_states)
            with torch.no_grad():
                next_q_values, _, _ = self.target_net(next_state_batch_encoded)
                next_state_values[non_final_mask] = next_q_values.max(1)[0]
        
        expected_q = reward_batch + (self.gamma * next_state_values)
        loss = F.smooth_l1_loss(current_q_selected[valid_action_mask], expected_q[valid_action_mask])
        
        self.training_net.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

        return loss.item(), 0  # No spike counting here, moved to select_action

    def update_target_network(self):
        self.target_net.load_state_dict(self.training_net.state_dict())
    
    def save_model(self, filename):
        torch.save(self.training_net.state_dict(), filename)

# Random Agent (Unchanged)
def random_agent(actions):
    return random.choice(actions)

# Hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TARGET_UPDATE = 10
MEMORY_CAPACITY = 100000
NUM_EPISODES = 20000
LOG_INTERVAL = 100

# DSNN Configuration
dsnn_config = {
    'architecture': [6 * 7, 128, 128, 7],
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.8,
    'weight_scale': 1.0,
    'batch_size': BATCH_SIZE,
    'threshold': 0.1,
    'simulation_time': 5,
    'learning_rate': 0.0005,
    'reset_potential': 0.0
}

# Initialization
env = ConnectX()
agent = DSQN(discount_factor=GAMMA, dsnn_config=dsnn_config)
memory = ReplayMemory()

# Global variables for spike tracking
total_spike_count = [0]
episode_count_for_avg_spikes = [0]

# Training Loop
steps_done = 0
total_loss_interval = 0
optimization_steps_interval = 0
interval_o_wins = 0
interval_x_wins = 0
interval_draws = 0
interval_total_moves = 0

for episode in range(NUM_EPISODES):
    state = env.reset()
    move_count = 0
    outcome = None

    for t in count():
        available_actions = env.get_available_actions()
        action = agent.select_action(state, available_actions, steps_done=steps_done, training=True)
        steps_done += 1
        
        next_state, reward, valid = env.make_move(action, 'p1')
        move_count += 1
        
        if not valid: outcome = 'loss'
        elif env.isDone: outcome = 'win' if reward == 1 else 'draw'
        
        memory.dump((state, action, reward, None if outcome else next_state))
        
        if outcome: break

        state_after_p1 = next_state
        available_actions = env.get_available_actions()
        action_p2 = random_agent(available_actions)
        next_state_after_p2, reward_p2, _ = env.make_move(action_p2, 'p2')
        move_count += 1
        
        if env.isDone:
            outcome = 'loss' if reward_p2 == 1 else 'draw'
        
        final_reward = -reward_p2 if outcome else env.reward['step']
        memory.dump((state, action, final_reward, None if outcome else next_state_after_p2))

        state = next_state_after_p2
        
        if outcome: break

    if outcome == 'win': interval_o_wins += 1
    elif outcome == 'loss': interval_x_wins += 1
    elif outcome == 'draw': interval_draws += 1
    interval_total_moves += move_count
    
    loss, _ = agent.optimize_model(memory, episode)
    if loss > 0:
        total_loss_interval += loss
        optimization_steps_interval += 1
        
    if episode % TARGET_UPDATE == 0:
        agent.update_target_network()

    if (episode + 1) % LOG_INTERVAL == 0 or episode == NUM_EPISODES - 1:
        total_games_in_interval = episode % LOG_INTERVAL + 1 if episode == NUM_EPISODES - 1 and (episode + 1) % LOG_INTERVAL != 0 else LOG_INTERVAL
        if total_games_in_interval > 0:
            o_win_rate = interval_o_wins / total_games_in_interval
            o_loss_rate = interval_x_wins / total_games_in_interval
            o_draw_rate = interval_draws / total_games_in_interval
            o_win_draw_rate = o_win_rate + o_draw_rate
            interval_avg_moves = interval_total_moves / total_games_in_interval
            avg_loss = total_loss_interval / optimization_steps_interval if optimization_steps_interval > 0 else 0
            average_spikes = total_spike_count[0] / episode_count_for_avg_spikes[0] if episode_count_for_avg_spikes[0] > 0 else 0

            print(f"\n--- Episode {episode + 1}/{NUM_EPISODES} ---")
            print(f"Training Summary (last {total_games_in_interval} episodes, 'O' perspective):")
            print(f" 'O' Wins: {interval_o_wins}, Losses: {interval_x_wins}, Draws: {interval_draws}")
            print(f" Win Rate: {o_win_rate:.3f}, Loss Rate: {o_loss_rate:.3f}, Draw Rate: {o_draw_rate:.3f}, Win+Draw Rate: {o_win_draw_rate:.3f}")
            print(f" Avg Moves per game: {interval_avg_moves:.1f}")
            print(f" Avg Loss: {avg_loss:.4f}")
            print(f" Average Spikes per Move: {average_spikes:.2f}")

            wandb.log({
                "Episode": episode + 1,
                "Wins": interval_o_wins,
                "Losses": interval_x_wins,
                "Draws": interval_draws,
                "Win Rate": o_win_rate,
                "Loss Rate": o_loss_rate,
                "Draw Rate": o_draw_rate,
                "Win+Draw Rate": o_win_draw_rate,
                "Interval Avg Moves": interval_avg_moves,
                "Average Loss": avg_loss,
                "Average Spikes per Move": average_spikes
            })
        
        total_loss_interval, optimization_steps_interval = 0, 0
        interval_o_wins, interval_x_wins, interval_draws, interval_total_moves = 0, 0, 0, 0
        total_spike_count[0], episode_count_for_avg_spikes[0] = 0, 0

wandb.finish()
print("\nTraining complete.")
model_save_path = "../saved_models/c4_count-rate.pth"
# Save final model
torch.save({
    'training_net': agent.training_net.state_dict(),
    'target_net': agent.target_net.state_dict()
}, model_save_path)
print("Trained model saved to '../saved_models/c4_count-rate.pth'")


# os.makedirs("saved_models", exist_ok=True)
# agent.save_model("saved_models/connect4_dsqn_rate_experiment.pth")
# print("Model saved to 'saved_models/connect4_dsqn_rate_experiment.pth'")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu

--- Episode 100/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 58, Losses: 41, Draws: 1
 Win Rate: 0.580, Loss Rate: 0.410, Draw Rate: 0.010, Win+Draw Rate: 0.590
 Avg Moves per game: 20.4
 Avg Loss: 4.6411
 Average Spikes per Move: 112.80

--- Episode 200/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 58, Losses: 42, Draws: 0
 Win Rate: 0.580, Loss Rate: 0.420, Draw Rate: 0.000, Win+Draw Rate: 0.580
 Avg Moves per game: 21.7
 Avg Loss: 5.8849
 Average Spikes per Move: 117.10

--- Episode 300/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 57, Losses: 43, Draws: 0
 Win Rate: 0.570, Loss Rate: 0.430, Draw Rate: 0.000, Win+Draw Rate: 0.570
 Avg Moves per game: 20.5
 Avg Loss: 6.5587
 Average Spikes per Move: 119.36

--- Episode 400/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 65, Losses: 35, Draws: 0
 Win Rate: 0.650, Loss Rate: 0.350, Draw Rate: 0.000, 

Average Loss,█▆▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Average Spikes per Move,▇▅▂▂▃▁▂▂▂▁▂▂▂▁▂▃▂▂▂▄▄▄▄▃▄▅▆▆▇▆▆▇▇▇███▇▇▆
Draw Rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Draws,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Episode,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
Interval Avg Moves,█▇▇▅▄▅▄▂▃▄▄▃▃▃▃▂▄▃▃▃▄▁▃▄▃▃▂▅▂▃▂▂▃▄▃▄▃▃▃▂
Loss Rate,▇█▆▄▄▄▂▅▃▃▆▃▂▄▂▄▄▂▅▅▂▂▃▄▆▆▄▂▅▂▅▂▁▁▄▄▄▄▃▄
Losses,█▄▆▅▅▆▃▄▄▄▄▄▄▄▅█▅▅▄▄▃▄▂▆▂▃▅▄▅▇▁▃▂▃▅▅▅▃▂▃
Win Rate,▁▃▆▃▄▄▃▆▄▆▆▄█▅▆█▆▅▅▆▆▆▄▄▃█▅▆▃▆▄▂▆▇▃█▅▆▆▇
Win+Draw Rate,▁▃▁▃▆▇▅▆▇▆▅▆▇█▇▆▆▆▅▅▆█▇▆▇▆█▅▇█▆▇▇▇▆▅█▆▆▇
+1,...



Training complete.
Trained model saved to '../saved_models/c4_count-rate.pth'


In [1]:
###Evaluation code for count rate coding
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Connect Four Environment
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self, visualize=False):
        if not visualize:
            return
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id): return True
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id): return True
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]): return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]): return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

# DQN Agent
class DQN(nn.Module):
    def __init__(self, outputs=7, height=6, width=7):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(height * width, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, outputs)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class DQNAgent:
    def __init__(self, discount_factor=0.99, epsilon=0.1):
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.training_network = DQN().to(device)

    def observe(self, state, action_space=None):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = self.training_network(state_tensor).detach().cpu().numpy().ravel()
        mac_count = (6 * 7 * 128) + (128 * 128) + (128 * 7)  # 5376 + 16384 + 896 = 22656 MACs
        if action_space is not None:
            valid_q = [(q_values[a], a) for a in action_space]
            action = max(valid_q, key=lambda x: x[0])[1]
        else:
            action = np.argmax(q_values)
        return action, mac_count

    def load_model(self, path):
        try:
            self.training_network.load_state_dict(torch.load(path, map_location=device))
            self.training_network.eval()
            print(f"DQN model loaded from {path}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DQN model file {path} not found")

# Surrogate Gradient Spike Function
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        mem_rec = []
        ac_count = 0
        internal_mac_count = 0
        
        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                    num_spikes = torch.sum(input_t > 0).item()
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])
                    num_spikes = torch.sum(spk[l-1][-1]).item()
                ac_count += num_spikes * self.weights[l].size(1)
                
                num_neurons = self.weights[l].size(1)
                internal_mac_count += num_neurons * 2
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    internal_mac_count += num_neurons * 2 * torch.mean(spk_current).item()
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        
        q_values = mem[-1]
        return q_values, mem_rec, spk, ac_count, internal_mac_count

# DSQN Agent
class DSQN:
    def __init__(self, discount_factor=0.99, dsnn_config=None):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.board_size = 6 * 7

    def rate_encode_single(self, state):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        encoded = torch.zeros(1, self.simulation_time, self.board_size, device=device)
        state_flat = state_tensor.flatten()
        spike_probs = torch.zeros_like(state_flat, device=device)
        spike_probs[state_flat == 1] = 3.0 / self.simulation_time  # Probability 3/5 for Player 1
        spike_probs[state_flat == 2] = 1.0  # Probability 1 for Player 2
        spike_probs_over_time = spike_probs.unsqueeze(0).expand(self.simulation_time, -1)  # Shape [5, 42]
        encoded[0] = torch.bernoulli(spike_probs_over_time)
        return encoded

    def observe(self, state, action_space=None):
        with torch.no_grad():
            state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
            encoded_state = self.rate_encode_single(state_tensor)
            q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
            if action_space is not None:
                valid_q = [(q_values[a], a) for a in action_space]
                action = max(valid_q, key=lambda x: x[0])[1]
            else:
                action = np.argmax(q_values)
            return action, mem_rec, all_spikes, ac_count, internal_mac_count

    # def load_model(self, filename):
    #     try:
    #         checkpoint = torch.load(filename, map_location=device)
    #         self.training_net.load_state_dict(checkpoint)
    #         self.target_net.load_state_dict(checkpoint)  # Copy to target_net for consistency
    #         self.training_net.eval()
    #         self.target_net.eval()
    #         print(f"DSQN model loaded from {filename}")
    #     except FileNotFoundError:
    #         raise FileNotFoundError(f"DSQN model file {filename} not found")
    #     except Exception as e:
    #         raise RuntimeError(f"Failed to load DSQN model from {filename}: {str(e)}")

    # In the EVALUATION script, inside the DSQN class

    def load_model(self, filename):
        try:
            # Load the entire dictionary that was saved
            checkpoint = torch.load(filename, map_location=device)

            # Check if the expected key exists
            if 'training_net' not in checkpoint:
                # This handles the case where you might have saved just the state_dict directly
                print("Loading model directly from state_dict.")
                self.training_net.load_state_dict(checkpoint)
            else:
                # Load the state_dict from the correct key
                print("Loading model from 'training_net' key in checkpoint.")
                self.training_net.load_state_dict(checkpoint['training_net'])
            
            # It's good practice to also load the target_net or at least sync it
            self.target_net.load_state_dict(self.training_net.state_dict())

            # Set models to evaluation mode
            self.training_net.eval()
            self.target_net.eval()
            print(f"DSQN model loaded successfully from {filename}")

        except FileNotFoundError:
            raise FileNotFoundError(f"DSQN model file {filename} not found")
        except Exception as e:
            raise RuntimeError(f"Failed to load DSQN model from {filename}: {str(e)}")

# Analysis Functions
def analyze_decision_stabilization(mem_rec, simulation_time):
    decision_times = []
    final_decision = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        stabilized = False
        for t in range(simulation_time):
            current_decision = torch.argmax(mem_rec[t], dim=1)[b]
            if current_decision == final_decision[b]:
                stable = True
                for t_next in range(t, simulation_time):
                    if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
                        stable = False
                        break
                if stable:
                    decision_times.append(t + 1)
                    stabilized = True
                    break
        if not stabilized:
            decision_times.append(simulation_time)
    return decision_times

def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
    total_spikes = 0
    total_neurons = sum(architecture[1:-1])  # 128 + 128 = 256
    spike_counts = []
    
    for t in range(len(all_spikes)):
        for l in range(len(all_spikes[t])):
            if l < len(architecture) - 1:
                spikes = all_spikes[t][l]
                spike_count = torch.sum(spikes).item()
                total_spikes += spike_count
                spike_counts.append(spike_count)
    
    sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
    return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count

# Utility Functions
def random_move(env):
    return random.choice(env.get_available_actions())

def play_game(agent, env, agent_first=True, visualize=False, collect_analysis=False):
    state = env.reset()
    current_player = 'p1' if agent_first else 'p2'
    agent_symbol = 'O' if agent_first else 'O'
    random_symbol = 'X' if agent_first else 'X'

    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if collect_analysis else None

    if visualize:
        print("\n=== New Game ===")
        print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
        env.render(visualize=True)

    while not env.isDone:
        if current_player == 'p1':
            if agent_first:
                action_space = env.get_available_actions()
                if collect_analysis and isinstance(agent, DSQN):
                    action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(state, action_space)
                    total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                    analysis_data['total_spikes'].append(total_spikes)
                    analysis_data['sparsity'].append(sparsity)
                    analysis_data['ac_counts'].append(ac_count)
                    analysis_data['internal_mac_counts'].append(internal_mac_count)
                    decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                    analysis_data['decision_times'].extend(decision_times)
                elif collect_analysis and isinstance(agent, DQNAgent):
                    action, mac_count = agent.observe(state, action_space)
                    analysis_data['mac_counts'].append(mac_count)
                else:
                    action = agent.observe(state, action_space)[0]
                next_state, reward, valid = env.make_move(action, 'p1')
                if not valid:
                    if visualize:
                        print(f"Invalid move by Agent ({agent_symbol})!")
                    return 'random', analysis_data
                state = next_state
                if visualize:
                    print(f"Agent ({agent_symbol}) played at column {action}")
                    env.render(visualize=True)
            else:
                action = random_move(env)
                next_state, reward, _ = env.make_move(action, 'p1')
                state = next_state
                if visualize:
                    print(f"Random ({random_symbol}) played at column {action}")
                    env.render(visualize=True)
        else:
            action = random_move(env)
            next_state, reward, _ = env.make_move(action, 'p2')
            state = next_state
            if visualize:
                print(f"Random ({random_symbol}) played at column {action}")
                env.render(visualize=True)

        if env.isDone:
            if visualize:
                if reward == 0.5:
                    print("Game ended in a draw!")
                elif (agent_first and reward == 1 and current_player == 'p1'):
                    print(f"Agent ({agent_symbol}) wins!")
                else:
                    print(f"Random ({random_symbol}) wins!")
            if reward == 0.5:
                return 'draw', analysis_data
            elif (agent_first and reward == 1 and current_player == 'p1'):
                return 'agent', analysis_data
            else:
                return 'random', analysis_data

        current_player = 'p2' if current_player == 'p1' else 'p1'

def test_agent_vs_random(agent, agent_name, env, num_games=100, visualize_all=False):
    results_agent_first = {'agent': 0, 'random': 0, 'draw': 0}
    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if isinstance(agent, DSQN) or isinstance(agent, DQNAgent) else None

    # Agent goes first ('O')
    if isinstance(agent, DQNAgent):
        try:
            agent.load_model("../saved_models/connect4_dqn_first_final.pth")
        except FileNotFoundError:
            print("DQN model file not found. Skipping DQN evaluation.")
            return
    elif isinstance(agent, DSQN):
        try:
            agent.load_model("../saved_models/c4_count-rate.pth")
        except FileNotFoundError:
            print("DSQN model file not found. Please run the training code to generate 'saved_models/connect4_dsqn_rate_experiment.pth'.")
            return
    for i in range(num_games):
        visualize = visualize_all
        winner, game_analysis = play_game(agent, env, agent_first=True, visualize=visualize, collect_analysis=True)
        results_agent_first[winner] += 1
        if game_analysis:
            analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
            analysis_data['sparsity'].extend(game_analysis['sparsity'])
            analysis_data['decision_times'].extend(game_analysis['decision_times'])
            analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
            analysis_data['mac_counts'].extend(game_analysis['mac_counts'])
            analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

    # Print game results
    print(f"\n{agent_name} vs. Random Agent - Final Results")
    print("="*50)
    print(f"{agent_name} goes first ({agent_name} as O, Random as X):")
    print(f"Wins ({agent_name}): {results_agent_first['agent']}, Losses: {results_agent_first['random']}, Draws: {results_agent_first['draw']}")
    print(f"Win rate: {results_agent_first['agent'] / num_games:.2%}, "
          f"Loss rate: {results_agent_first['random'] / num_games:.2%}, "
          f"Draw rate: {results_agent_first['draw'] / num_games:.2%}")

    # Print analysis results
    if isinstance(agent, DSQN) and analysis_data['total_spikes']:
        print(f"\nDSQN Analysis Results")
        print("="*50)
        print(f"Average Total Spikes per Decision: {float(np.mean(analysis_data['total_spikes'])):.2f} "
              f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
        print(f"Average Sparsity per Decision: {float(np.mean(analysis_data['sparsity'])):.2%} "
              f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
        print(f"Average ACs per Decision (spike-triggered additions): {float(np.mean(analysis_data['ac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
        print(f"Average Internal State Update MACs per Decision: {float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")
        decision_time_counts = np.bincount(analysis_data['decision_times'], minlength=agent.simulation_time + 1)[1:]
        print(f"Decision Stabilization Times (over all decisions):")
        for t in range(agent.simulation_time):
            print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
                  f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")
        total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
        total_neurons = 256  # 128 + 128
        total_possible_spikes = total_neurons * agent.simulation_time
        f_r = total_spikes_avg / total_possible_spikes
        avg_ac = float(np.mean(analysis_data['ac_counts']))
        avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))
        ann_energy = 22656 * 31  # DQN MACs * E_MAC (E_AC = 1)
        snn_energy = avg_ac + avg_internal_mac * 31  # E_SNN = AC * E_AC + Internal_MAC * E_MAC
        energy_ratio_detailed = snn_energy / ann_energy
        energy_savings_detailed = (1 - energy_ratio_detailed) * 100
        energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
        energy_savings_simplified = (1 - energy_ratio_simplified) * 100
        print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): {energy_ratio_simplified:.4f}")
        print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
        print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): {energy_ratio_detailed:.4f}")
        print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")
    elif isinstance(agent, DQNAgent) and analysis_data['mac_counts']:
        print(f"\nDQN Analysis Results")
        print("="*50)
        print(f"Average MACs per Decision: {float(np.mean(analysis_data['mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['mac_counts'])):.2f})")

if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    dsnn_config = {
        'architecture': [6 * 7, 128, 128, 7],
        'seed': 42,
        'alpha': 0.9,
        'beta': 0.8,
        'weight_scale': 1.0,
        'batch_size': 128,
        'threshold': 0.1,
        'simulation_time': 5,
        'learning_rate': 0.0005,
        'reset_potential': 0.0
    }

    env = ConnectX()
    dqn_agent = DQNAgent()
    dsqn_agent = DSQN(dsnn_config=dsnn_config)

    print("\nTesting DQN vs. Random Agent")
    test_agent_vs_random(dqn_agent, "DQN", env, num_games=100, visualize_all=False)
    
    print("\nTesting DSQN vs. Random Agent")
    test_agent_vs_random(dsqn_agent, "DSQN", env, num_games=100, visualize_all=False)



Testing DQN vs. Random Agent
DQN model loaded from ../saved_models/connect4_dqn_first_final.pth

DQN vs. Random Agent - Final Results
DQN goes first (DQN as O, Random as X):
Wins (DQN): 82, Losses: 18, Draws: 0
Win rate: 82.00%, Loss rate: 18.00%, Draw rate: 0.00%

DQN Analysis Results
Average MACs per Decision: 22656.00 (Std: 0.00)

Testing DSQN vs. Random Agent
Loading model from 'training_net' key in checkpoint.
DSQN model loaded successfully from ../saved_models/c4_count-rate.pth


/tmp/ipykernel_34515/3598676122.py:205: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_tensor = torch.tensor(state, dtype=torch.float32, device=device)



DSQN vs. Random Agent - Final Results
DSQN goes first (DSQN as O, Random as X):
Wins (DSQN): 57, Losses: 43, Draws: 0
Win rate: 57.00%, Loss rate: 43.00%, Draw rate: 0.00%

DSQN Analysis Results
Average Total Spikes per Decision: 365.03 (Std: 136.79)
Average Sparsity per Decision: 52.47% (Std: 17.81%)
Average ACs per Decision (spike-triggered additions): 46766.67 (Std: 18276.65)
Average Internal State Update MACs per Decision: 3862.28 (Std: 461.05)
Decision Stabilization Times (over all decisions):
  Time Step 1: 775 decisions (94.05%)
  Time Step 2: 7 decisions (0.85%)
  Time Step 3: 9 decisions (1.09%)
  Time Step 4: 18 decisions (2.18%)
  Time Step 5: 15 decisions (1.82%)
Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): 0.0460
Simplified Energy Savings: 95.4%
Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): 0.2371
Detailed Energy Savings: 76.3%


In [1]:
##rate coding vectorized and optimized with not original parameters
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display
from itertools import count
import wandb
import os

# Initialize WandB
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="count-rate_tuned_c4")

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Connect Four Environment Class (Unchanged)
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self):
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id): return True
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id): return True
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]): return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]): return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            print('Move is invalid')
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

# Replay Memory Class (Unchanged)
class ReplayMemory:
    def __init__(self, capacity=100000):
        self.memory = []
        self.capacity = capacity
        self.position = 0

    def dump(self, transition):
        if len(self.memory) < self.capacity: self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

# Surrogate Gradient Spike Function (Unchanged)
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN) Class (Unchanged)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size,
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        torch.manual_seed(seed)
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        mem_rec = []
        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        q_values = mem_rec[-1]
        return q_values, mem_rec, spk

# DSQN Agent Class (Updated)
class DSQN:
    def __init__(self, discount_factor, dsnn_config):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        self.board_size = 6 * 7

    def rate_encode_single(self, state):
        """Original iterative encoding for a single state (used in select_action)."""
        state_tensor = torch.tensor(state, dtype=torch.float, device=device)
        encoded = torch.zeros(self.simulation_time, self.board_size, device=device)
        state_flat = state_tensor.flatten()
        for pos, val in enumerate(state_flat):
            if val == 0: spike_prob = 0.0
            elif val == 1: spike_prob = 3.0 / self.simulation_time
            elif val == 2: spike_prob = 1.0
            encoded[:, pos] = torch.bernoulli(torch.full((self.simulation_time,), spike_prob, device=device))
        return encoded.unsqueeze(0)

    def rate_encode_batch(self, states_batch):
        """Vectorized rate encoding for a batch of states."""
        batch_size = states_batch.size(0)
        states_flat = states_batch.view(batch_size, -1)
        spike_probs = torch.zeros_like(states_flat, dtype=torch.float, device=device)
        spike_probs[states_flat == 1] = 3.0 / self.simulation_time
        spike_probs[states_flat == 2] = 1.0
        spike_probs_over_time = spike_probs.unsqueeze(2).expand(-1, -1, self.simulation_time)
        encoded = torch.bernoulli(spike_probs_over_time)
        return encoded.permute(0, 2, 1)

    def select_action(self, state, available_actions, training=True, steps_done=None):
        encoded_state = self.rate_encode_single(state)
        if training:
            if steps_done is None: raise ValueError("steps_done is required")
            eps_threshold = EPS_END + (EPS_START - EPS_END) * np.exp(-steps_done / EPS_DECAY)
        else:
            eps_threshold = 0
        if random.random() > eps_threshold:
            with torch.no_grad():
                q_values, _, spk = self.training_net(encoded_state)
                # Sum spikes across hidden layers for the last timestep, as in Tic-Tac-Toe
                step_spike_count = sum(layer_spikes[-1].sum().item() for layer_spikes in spk[:-1] if layer_spikes)
                total_spike_count[0] += step_spike_count
                episode_count_for_avg_spikes[0] += 1
                q_values = q_values.flatten()
                valid_q = q_values[available_actions]
                return available_actions[torch.argmax(valid_q).item()]
        return random.choice(available_actions)

    def optimize_model(self, memory, episode):
        if len(memory) < self.batch_size:
            return 0.0, 0
        
        transitions = memory.sample(self.batch_size)
        state_batch, action_batch, reward_batch, next_state_batch = zip(*transitions)
        
        state_batch = torch.tensor(np.array(state_batch), dtype=torch.float, device=device)
        action_batch = torch.tensor(action_batch, dtype=torch.long, device=device)
        reward_batch = torch.tensor(reward_batch, dtype=torch.float, device=device)
        non_final_mask = torch.tensor([s is not None for s in next_state_batch], device=device)
        non_final_next_states = [s for s in next_state_batch if s is not None]
        non_final_next_states = torch.tensor(np.array(non_final_next_states), dtype=torch.float, device=device) if non_final_next_states else None
        
        state_batch_encoded = self.rate_encode_batch(state_batch)
        current_q, _, _ = self.training_net(state_batch_encoded)
        
        valid_action_mask = action_batch != -1
        current_q_selected = torch.zeros(self.batch_size, device=device)
        if valid_action_mask.any():
            valid_indices = valid_action_mask.nonzero(as_tuple=True)[0]
            valid_actions = action_batch[valid_action_mask].unsqueeze(1)
            current_q_selected[valid_indices] = current_q[valid_indices].gather(1, valid_actions).squeeze(1)
        
        next_state_values = torch.zeros(self.batch_size, device=device)
        if non_final_next_states is not None:
            next_state_batch_encoded = self.rate_encode_batch(non_final_next_states)
            with torch.no_grad():
                next_q_values, _, _ = self.target_net(next_state_batch_encoded)
                next_state_values[non_final_mask] = next_q_values.max(1)[0]
        
        expected_q = reward_batch + (self.gamma * next_state_values)
        loss = F.smooth_l1_loss(current_q_selected[valid_action_mask], expected_q[valid_action_mask])
        
        self.training_net.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

        return loss.item(), 0  # No spike counting here, moved to select_action

    def update_target_network(self):
        self.target_net.load_state_dict(self.training_net.state_dict())
    
    def save_model(self, filename):
        torch.save(self.training_net.state_dict(), filename)

# Random Agent (Unchanged)
def random_agent(actions):
    return random.choice(actions)

# Hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 20000
TARGET_UPDATE = 10
MEMORY_CAPACITY = 100000
NUM_EPISODES = 20000
LOG_INTERVAL = 100

# DSNN Configuration
dsnn_config = {
    'architecture': [6 * 7, 128, 128, 7],
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.8,
    'weight_scale': 1.0,
    'batch_size': BATCH_SIZE,
    'threshold': 0.1,
    'simulation_time': 5,
    'learning_rate': 0.00025,
    'reset_potential': 0.0
}

# Initialization
env = ConnectX()
agent = DSQN(discount_factor=GAMMA, dsnn_config=dsnn_config)
memory = ReplayMemory()

# Global variables for spike tracking
total_spike_count = [0]
episode_count_for_avg_spikes = [0]

# Training Loop
steps_done = 0
total_loss_interval = 0
optimization_steps_interval = 0
interval_o_wins = 0
interval_x_wins = 0
interval_draws = 0
interval_total_moves = 0

for episode in range(NUM_EPISODES):
    state = env.reset()
    move_count = 0
    outcome = None

    for t in count():
        available_actions = env.get_available_actions()
        action = agent.select_action(state, available_actions, steps_done=steps_done, training=True)
        steps_done += 1
        
        next_state, reward, valid = env.make_move(action, 'p1')
        move_count += 1
        
        if not valid: outcome = 'loss'
        elif env.isDone: outcome = 'win' if reward == 1 else 'draw'
        
        memory.dump((state, action, reward, None if outcome else next_state))
        
        if outcome: break

        state_after_p1 = next_state
        available_actions = env.get_available_actions()
        action_p2 = random_agent(available_actions)
        next_state_after_p2, reward_p2, _ = env.make_move(action_p2, 'p2')
        move_count += 1
        
        if env.isDone:
            outcome = 'loss' if reward_p2 == 1 else 'draw'
        
        final_reward = -reward_p2 if outcome else env.reward['step']
        memory.dump((state, action, final_reward, None if outcome else next_state_after_p2))

        state = next_state_after_p2
        
        if outcome: break

    if outcome == 'win': interval_o_wins += 1
    elif outcome == 'loss': interval_x_wins += 1
    elif outcome == 'draw': interval_draws += 1
    interval_total_moves += move_count
    
    loss, _ = agent.optimize_model(memory, episode)
    if loss > 0:
        total_loss_interval += loss
        optimization_steps_interval += 1
        
    if episode % TARGET_UPDATE == 0:
        agent.update_target_network()

    if (episode + 1) % LOG_INTERVAL == 0 or episode == NUM_EPISODES - 1:
        total_games_in_interval = episode % LOG_INTERVAL + 1 if episode == NUM_EPISODES - 1 and (episode + 1) % LOG_INTERVAL != 0 else LOG_INTERVAL
        if total_games_in_interval > 0:
            o_win_rate = interval_o_wins / total_games_in_interval
            o_loss_rate = interval_x_wins / total_games_in_interval
            o_draw_rate = interval_draws / total_games_in_interval
            o_win_draw_rate = o_win_rate + o_draw_rate
            interval_avg_moves = interval_total_moves / total_games_in_interval
            avg_loss = total_loss_interval / optimization_steps_interval if optimization_steps_interval > 0 else 0
            average_spikes = total_spike_count[0] / episode_count_for_avg_spikes[0] if episode_count_for_avg_spikes[0] > 0 else 0

            print(f"\n--- Episode {episode + 1}/{NUM_EPISODES} ---")
            print(f"Training Summary (last {total_games_in_interval} episodes, 'O' perspective):")
            print(f" 'O' Wins: {interval_o_wins}, Losses: {interval_x_wins}, Draws: {interval_draws}")
            print(f" Win Rate: {o_win_rate:.3f}, Loss Rate: {o_loss_rate:.3f}, Draw Rate: {o_draw_rate:.3f}, Win+Draw Rate: {o_win_draw_rate:.3f}")
            print(f" Avg Moves per game: {interval_avg_moves:.1f}")
            print(f" Avg Loss: {avg_loss:.4f}")
            print(f" Average Spikes per Move: {average_spikes:.2f}")

            wandb.log({
                "Episode": episode + 1,
                "Wins": interval_o_wins,
                "Losses": interval_x_wins,
                "Draws": interval_draws,
                "Win Rate": o_win_rate,
                "Loss Rate": o_loss_rate,
                "Draw Rate": o_draw_rate,
                "Win+Draw Rate": o_win_draw_rate,
                "Interval Avg Moves": interval_avg_moves,
                "Average Loss": avg_loss,
                "Average Spikes per Move": average_spikes
            })
        
        total_loss_interval, optimization_steps_interval = 0, 0
        interval_o_wins, interval_x_wins, interval_draws, interval_total_moves = 0, 0, 0, 0
        total_spike_count[0], episode_count_for_avg_spikes[0] = 0, 0

wandb.finish()
print("\nTraining complete.")

model_save_path = "../saved_models/c4_count-rate_tuned.pth"
# Save final model
torch.save({
    'training_net': agent.training_net.state_dict(),
    'target_net': agent.target_net.state_dict()
}, model_save_path)
print("Trained model saved to '../saved_models/c4_count-rate_tuned.pth'")

# os.makedirs("saved_models", exist_ok=True)
# agent.save_model("saved_models/connect4_dsqn_rate_optimized.pth")
# print("Model saved to 'saved_models/connect4_dsqn_rate_optimized.pth'")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu

--- Episode 100/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 53, Losses: 47, Draws: 0
 Win Rate: 0.530, Loss Rate: 0.470, Draw Rate: 0.000, Win+Draw Rate: 0.530
 Avg Moves per game: 21.1
 Avg Loss: 5.5150
 Average Spikes per Move: 109.43

--- Episode 200/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 55, Losses: 45, Draws: 0
 Win Rate: 0.550, Loss Rate: 0.450, Draw Rate: 0.000, Win+Draw Rate: 0.550
 Avg Moves per game: 21.6
 Avg Loss: 5.7359
 Average Spikes per Move: 115.19

--- Episode 300/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 60, Losses: 39, Draws: 1
 Win Rate: 0.600, Loss Rate: 0.390, Draw Rate: 0.010, Win+Draw Rate: 0.610
 Avg Moves per game: 21.9
 Avg Loss: 6.8410
 Average Spikes per Move: 125.88

--- Episode 400/20000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 61, Losses: 39, Draws: 0
 Win Rate: 0.610, Loss Rate: 0.390, Draw Rate: 0.000, 

Average Loss,█▄▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Average Spikes per Move,▁▇█▆▇▆▆▅▅▅▆▆▆▅▅▅▄▆▅▄▅▆▄▄▅▄▄▅▅▆▅▅▅▅▅▅▅▆▆▅
Draw Rate,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Draws,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Episode,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███
Interval Avg Moves,██▇▆▄▄▄▃▃▃▂▃▃▃▂▂▂▂▃▃▂▃▂▂▂▂▂▂▁▁▂▂▂▁▂▁▂▂▂▁
Loss Rate,█▇▇▇▆▄▅▃▅▄▂▂▅▃▃▃▃▃▂▁▂▃▃▃▃▂▂▂▁▃▃▁▂▂▃▂▃▃▃▃
Losses,█▆▇▆▅▆▅▄▅▃▃▃▄▁▃▂▁▂▃▂▂▃▂▂▂▂▁▃▁▃▂▂▁▃▃▂▃▂▁▃
Win Rate,▁▁▅▂▅▄▅▂▄▅▅▆▆▇▅▇▇▅▆▆▅▇▅▇▆██▆▅▆▇▇▅▆▅▅▇▆▆█
Win+Draw Rate,▁▅▁▄▄▆▄▅▅▅▅▅▆▆▅▅▇▆▆▆▇▇█▆▅▇▇▆▅▇▆▇▅▆▆▅▇▆█▆
+1,...



Training complete.
Trained model saved to '../saved_models/c4_count-rate_tuned.pth'


In [1]:
###Evaluation code for count rate coding
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Connect Four Environment
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self, visualize=False):
        if not visualize:
            return
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id): return True
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id): return True
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]): return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]): return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

# DQN Agent
class DQN(nn.Module):
    def __init__(self, outputs=7, height=6, width=7):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(height * width, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, outputs)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class DQNAgent:
    def __init__(self, discount_factor=0.99, epsilon=0.1):
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.training_network = DQN().to(device)

    def observe(self, state, action_space=None):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = self.training_network(state_tensor).detach().cpu().numpy().ravel()
        mac_count = (6 * 7 * 128) + (128 * 128) + (128 * 7)  # 5376 + 16384 + 896 = 22656 MACs
        if action_space is not None:
            valid_q = [(q_values[a], a) for a in action_space]
            action = max(valid_q, key=lambda x: x[0])[1]
        else:
            action = np.argmax(q_values)
        return action, mac_count

    def load_model(self, path):
        try:
            self.training_network.load_state_dict(torch.load(path, map_location=device))
            self.training_network.eval()
            print(f"DQN model loaded from {path}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DQN model file {path} not found")

# Surrogate Gradient Spike Function
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        mem_rec = []
        ac_count = 0
        internal_mac_count = 0
        
        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                    num_spikes = torch.sum(input_t > 0).item()
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])
                    num_spikes = torch.sum(spk[l-1][-1]).item()
                ac_count += num_spikes * self.weights[l].size(1)
                
                num_neurons = self.weights[l].size(1)
                internal_mac_count += num_neurons * 2
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    internal_mac_count += num_neurons * 2 * torch.mean(spk_current).item()
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        
        q_values = mem[-1]
        return q_values, mem_rec, spk, ac_count, internal_mac_count

# DSQN Agent
class DSQN:
    def __init__(self, discount_factor=0.99, dsnn_config=None):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.board_size = 6 * 7

    def rate_encode_single(self, state):
        state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
        encoded = torch.zeros(1, self.simulation_time, self.board_size, device=device)
        state_flat = state_tensor.flatten()
        spike_probs = torch.zeros_like(state_flat, device=device)
        spike_probs[state_flat == 1] = 3.0 / self.simulation_time  # Probability 3/5 for Player 1
        spike_probs[state_flat == 2] = 1.0  # Probability 1 for Player 2
        spike_probs_over_time = spike_probs.unsqueeze(0).expand(self.simulation_time, -1)  # Shape [5, 42]
        encoded[0] = torch.bernoulli(spike_probs_over_time)
        return encoded

    def observe(self, state, action_space=None):
        with torch.no_grad():
            state_tensor = torch.tensor(state, dtype=torch.float32, device=device)
            encoded_state = self.rate_encode_single(state_tensor)
            q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
            if action_space is not None:
                valid_q = [(q_values[a], a) for a in action_space]
                action = max(valid_q, key=lambda x: x[0])[1]
            else:
                action = np.argmax(q_values)
            return action, mem_rec, all_spikes, ac_count, internal_mac_count

    # def load_model(self, filename):
    #     try:
    #         checkpoint = torch.load(filename, map_location=device)
    #         self.training_net.load_state_dict(checkpoint)
    #         self.target_net.load_state_dict(checkpoint)  # Copy to target_net for consistency
    #         self.training_net.eval()
    #         self.target_net.eval()
    #         print(f"DSQN model loaded from {filename}")
    #     except FileNotFoundError:
    #         raise FileNotFoundError(f"DSQN model file {filename} not found")
    #     except Exception as e:
    #         raise RuntimeError(f"Failed to load DSQN model from {filename}: {str(e)}")

    # In the EVALUATION script, inside the DSQN class

    def load_model(self, filename):
        try:
            # Load the entire dictionary that was saved
            checkpoint = torch.load(filename, map_location=device)

            # Check if the expected key exists
            if 'training_net' not in checkpoint:
                # This handles the case where you might have saved just the state_dict directly
                print("Loading model directly from state_dict.")
                self.training_net.load_state_dict(checkpoint)
            else:
                # Load the state_dict from the correct key
                print("Loading model from 'training_net' key in checkpoint.")
                self.training_net.load_state_dict(checkpoint['training_net'])
            
            # It's good practice to also load the target_net or at least sync it
            self.target_net.load_state_dict(self.training_net.state_dict())

            # Set models to evaluation mode
            self.training_net.eval()
            self.target_net.eval()
            print(f"DSQN model loaded successfully from {filename}")

        except FileNotFoundError:
            raise FileNotFoundError(f"DSQN model file {filename} not found")
        except Exception as e:
            raise RuntimeError(f"Failed to load DSQN model from {filename}: {str(e)}")

# Analysis Functions
def analyze_decision_stabilization(mem_rec, simulation_time):
    decision_times = []
    final_decision = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        stabilized = False
        for t in range(simulation_time):
            current_decision = torch.argmax(mem_rec[t], dim=1)[b]
            if current_decision == final_decision[b]:
                stable = True
                for t_next in range(t, simulation_time):
                    if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
                        stable = False
                        break
                if stable:
                    decision_times.append(t + 1)
                    stabilized = True
                    break
        if not stabilized:
            decision_times.append(simulation_time)
    return decision_times

def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
    total_spikes = 0
    total_neurons = sum(architecture[1:-1])  # 128 + 128 = 256
    spike_counts = []
    
    for t in range(len(all_spikes)):
        for l in range(len(all_spikes[t])):
            if l < len(architecture) - 1:
                spikes = all_spikes[t][l]
                spike_count = torch.sum(spikes).item()
                total_spikes += spike_count
                spike_counts.append(spike_count)
    
    sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
    return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count

# Utility Functions
def random_move(env):
    return random.choice(env.get_available_actions())

def play_game(agent, env, agent_first=True, visualize=False, collect_analysis=False):
    state = env.reset()
    current_player = 'p1' if agent_first else 'p2'
    agent_symbol = 'O' if agent_first else 'O'
    random_symbol = 'X' if agent_first else 'X'

    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if collect_analysis else None

    if visualize:
        print("\n=== New Game ===")
        print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
        env.render(visualize=True)

    while not env.isDone:
        if current_player == 'p1':
            if agent_first:
                action_space = env.get_available_actions()
                if collect_analysis and isinstance(agent, DSQN):
                    action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(state, action_space)
                    total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                    analysis_data['total_spikes'].append(total_spikes)
                    analysis_data['sparsity'].append(sparsity)
                    analysis_data['ac_counts'].append(ac_count)
                    analysis_data['internal_mac_counts'].append(internal_mac_count)
                    decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                    analysis_data['decision_times'].extend(decision_times)
                elif collect_analysis and isinstance(agent, DQNAgent):
                    action, mac_count = agent.observe(state, action_space)
                    analysis_data['mac_counts'].append(mac_count)
                else:
                    action = agent.observe(state, action_space)[0]
                next_state, reward, valid = env.make_move(action, 'p1')
                if not valid:
                    if visualize:
                        print(f"Invalid move by Agent ({agent_symbol})!")
                    return 'random', analysis_data
                state = next_state
                if visualize:
                    print(f"Agent ({agent_symbol}) played at column {action}")
                    env.render(visualize=True)
            else:
                action = random_move(env)
                next_state, reward, _ = env.make_move(action, 'p1')
                state = next_state
                if visualize:
                    print(f"Random ({random_symbol}) played at column {action}")
                    env.render(visualize=True)
        else:
            action = random_move(env)
            next_state, reward, _ = env.make_move(action, 'p2')
            state = next_state
            if visualize:
                print(f"Random ({random_symbol}) played at column {action}")
                env.render(visualize=True)

        if env.isDone:
            if visualize:
                if reward == 0.5:
                    print("Game ended in a draw!")
                elif (agent_first and reward == 1 and current_player == 'p1'):
                    print(f"Agent ({agent_symbol}) wins!")
                else:
                    print(f"Random ({random_symbol}) wins!")
            if reward == 0.5:
                return 'draw', analysis_data
            elif (agent_first and reward == 1 and current_player == 'p1'):
                return 'agent', analysis_data
            else:
                return 'random', analysis_data

        current_player = 'p2' if current_player == 'p1' else 'p1'

def test_agent_vs_random(agent, agent_name, env, num_games=100, visualize_all=False):
    results_agent_first = {'agent': 0, 'random': 0, 'draw': 0}
    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if isinstance(agent, DSQN) or isinstance(agent, DQNAgent) else None

    # Agent goes first ('O')
    if isinstance(agent, DQNAgent):
        try:
            agent.load_model("../saved_models/connect4_dqn_first_final.pth")
        except FileNotFoundError:
            print("DQN model file not found. Skipping DQN evaluation.")
            return
    elif isinstance(agent, DSQN):
        try:
            agent.load_model("../saved_models/c4_count-rate_tuned.pth")
        except FileNotFoundError:
            print("DSQN model file not found. Please run the training code to generate 'saved_models/connect4_dsqn_rate_experiment.pth'.")
            return
    for i in range(num_games):
        visualize = visualize_all
        winner, game_analysis = play_game(agent, env, agent_first=True, visualize=visualize, collect_analysis=True)
        results_agent_first[winner] += 1
        if game_analysis:
            analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
            analysis_data['sparsity'].extend(game_analysis['sparsity'])
            analysis_data['decision_times'].extend(game_analysis['decision_times'])
            analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
            analysis_data['mac_counts'].extend(game_analysis['mac_counts'])
            analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

    # Print game results
    print(f"\n{agent_name} vs. Random Agent - Final Results")
    print("="*50)
    print(f"{agent_name} goes first ({agent_name} as O, Random as X):")
    print(f"Wins ({agent_name}): {results_agent_first['agent']}, Losses: {results_agent_first['random']}, Draws: {results_agent_first['draw']}")
    print(f"Win rate: {results_agent_first['agent'] / num_games:.2%}, "
          f"Loss rate: {results_agent_first['random'] / num_games:.2%}, "
          f"Draw rate: {results_agent_first['draw'] / num_games:.2%}")

    # Print analysis results
    if isinstance(agent, DSQN) and analysis_data['total_spikes']:
        print(f"\nDSQN Analysis Results")
        print("="*50)
        print(f"Average Total Spikes per Decision: {float(np.mean(analysis_data['total_spikes'])):.2f} "
              f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
        print(f"Average Sparsity per Decision: {float(np.mean(analysis_data['sparsity'])):.2%} "
              f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
        print(f"Average ACs per Decision (spike-triggered additions): {float(np.mean(analysis_data['ac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
        print(f"Average Internal State Update MACs per Decision: {float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")
        decision_time_counts = np.bincount(analysis_data['decision_times'], minlength=agent.simulation_time + 1)[1:]
        print(f"Decision Stabilization Times (over all decisions):")
        for t in range(agent.simulation_time):
            print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
                  f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")
        total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
        total_neurons = 256  # 128 + 128
        total_possible_spikes = total_neurons * agent.simulation_time
        f_r = total_spikes_avg / total_possible_spikes
        avg_ac = float(np.mean(analysis_data['ac_counts']))
        avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))
        ann_energy = 22656 * 31  # DQN MACs * E_MAC (E_AC = 1)
        snn_energy = avg_ac + avg_internal_mac * 31  # E_SNN = AC * E_AC + Internal_MAC * E_MAC
        energy_ratio_detailed = snn_energy / ann_energy
        energy_savings_detailed = (1 - energy_ratio_detailed) * 100
        energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
        energy_savings_simplified = (1 - energy_ratio_simplified) * 100
        print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): {energy_ratio_simplified:.4f}")
        print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
        print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): {energy_ratio_detailed:.4f}")
        print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")
    elif isinstance(agent, DQNAgent) and analysis_data['mac_counts']:
        print(f"\nDQN Analysis Results")
        print("="*50)
        print(f"Average MACs per Decision: {float(np.mean(analysis_data['mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['mac_counts'])):.2f})")

if __name__ == "__main__":
    random.seed(82)
    np.random.seed(82)
    torch.manual_seed(82)

    dsnn_config = {
        'architecture': [6 * 7, 128, 128, 7],
        'seed': 82,
        'alpha': 0.9,
        'beta': 0.8,
        'weight_scale': 1.0,
        'batch_size': 128,
        'threshold': 0.1,
        'simulation_time': 5,
        'learning_rate': 0.0005,
        'reset_potential': 0.0
    }

    env = ConnectX()
    dqn_agent = DQNAgent()
    dsqn_agent = DSQN(dsnn_config=dsnn_config)

    print("\nTesting DQN vs. Random Agent")
    test_agent_vs_random(dqn_agent, "DQN", env, num_games=100, visualize_all=False)
    
    print("\nTesting DSQN vs. Random Agent")
    test_agent_vs_random(dsqn_agent, "DSQN", env, num_games=100, visualize_all=False)



Testing DQN vs. Random Agent
DQN model loaded from ../saved_models/connect4_dqn_first_final.pth

DQN vs. Random Agent - Final Results
DQN goes first (DQN as O, Random as X):
Wins (DQN): 96, Losses: 4, Draws: 0
Win rate: 96.00%, Loss rate: 4.00%, Draw rate: 0.00%

DQN Analysis Results
Average MACs per Decision: 22656.00 (Std: 0.00)

Testing DSQN vs. Random Agent
Loading model from 'training_net' key in checkpoint.
DSQN model loaded successfully from ../saved_models/c4_count-rate_tuned.pth


/tmp/ipykernel_22285/1841791390.py:205: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_tensor = torch.tensor(state, dtype=torch.float32, device=device)



DSQN vs. Random Agent - Final Results
DSQN goes first (DSQN as O, Random as X):
Wins (DSQN): 90, Losses: 10, Draws: 0
Win rate: 90.00%, Loss rate: 10.00%, Draw rate: 0.00%

DSQN Analysis Results
Average Total Spikes per Decision: 329.30 (Std: 162.99)
Average Sparsity per Decision: 57.12% (Std: 21.22%)
Average ACs per Decision (spike-triggered additions): 38109.32 (Std: 19305.48)
Average Internal State Update MACs per Decision: 3745.79 (Std: 551.63)
Decision Stabilization Times (over all decisions):
  Time Step 1: 507 decisions (99.02%)
  Time Step 2: 0 decisions (0.00%)
  Time Step 3: 0 decisions (0.00%)
  Time Step 4: 5 decisions (0.98%)
  Time Step 5: 0 decisions (0.00%)
Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): 0.0415
Simplified Energy Savings: 95.9%
Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): 0.2196
Detailed Energy Savings: 78.0%
